# 05 - Missing and Duplicate Data

**Suggested time: 15 minutes**

Real extracts contain nulls, blank strings, inconsistent whitespace, and duplicate records. This lesson builds a small repeatable cleaning pipeline.

## Learning objectives

By the end of this notebook, you will be able to:

- distinguish null from an empty string;
- standardise basic text values;
- use `isNull`, `coalesce`, `fillna`, and `dropna`; and
- remove exact duplicates safely.

## Prerequisite recap

Notebook 04 introduced column expressions and conditional logic. We will combine those tools into a multi-step cleaning transformation.

In [ ]:
from pyspark.sql import functions as F

customers = spark.createDataFrame(
    [
        ('C001', ' Northwind Supplies ', 'orders@northwind.example', None, 'North'),
        ('C002', 'Contoso Retail', None, 'contact@contoso.example', 'West'),
        ('C003', 'Adventure Works', None, None, '   '),
        ('C004', 'Fabrikam Stores', 'sales@fabrikam.example', None, 'South'),
        ('C004', 'Fabrikam Stores', 'sales@fabrikam.example', None, 'South'),
        (None, 'Unidentified Customer', None, 'unknown@example', None),
    ],
    'customer_id STRING, customer_name STRING, work_email STRING, personal_email STRING, region STRING',
)
customers.show(truncate=False)

## Null is not the same as blank

`NULL` means no value is present. `''` and whitespace such as `'   '` are still strings. Standardise blank strings to null before applying null-handling rules.

In [ ]:
customers_standardised = (
    customers
    .withColumn('customer_name', F.trim(F.col('customer_name')))
    .withColumn(
        'region',
        F.when(F.trim(F.col('region')) == '', F.lit(None))
        .otherwise(F.trim(F.col('region'))),
    )
)
customers_standardised.show(truncate=False)

## Find missing values

Use `isNull()` and `isNotNull()` inside a filter to inspect records before deciding how to handle them.

In [ ]:
customers_standardised.filter(F.col('work_email').isNull()).show()
customers_standardised.filter(F.col('customer_id').isNotNull()).show()

## Choose fallbacks, fill values, and reject unusable rows

- `coalesce` returns the first non-null expression.
- `fillna` supplies a known replacement.
- `dropna` removes rows missing required fields.

A missing customer ID is rejected because it cannot be joined reliably.

In [ ]:
customer_contacts = (
    customers_standardised
    .withColumn('preferred_email', F.coalesce('work_email', 'personal_email'))
    .dropna(subset=['customer_id'])
    .fillna({'region': 'Unknown'})
)
customer_contacts.show(truncate=False)

## Remove duplicates deliberately

`dropDuplicates()` removes rows that are identical across all columns. Supplying key columns, such as `dropDuplicates(['customer_id'])`, keeps an arbitrary record when those records differ. Notebook 10 shows how to choose the latest record deterministically.

In [ ]:
customers_clean = customer_contacts.dropDuplicates()
customers_clean.show(truncate=False)

print('Rows before exact deduplication:', customer_contacts.count())
print('Rows after exact deduplication:', customers_clean.count())

## A small data-quality check

Cleaning does not imply that every optional field is populated. Count remaining records without any usable email so the business can decide what to do.

In [ ]:
missing_contact_count = customers_clean.filter(
    F.col('preferred_email').isNull()
).count()
print('Customers needing contact review:', missing_contact_count)

## Your turn

Create `customers_ready` directly from `customers`. Trim names and regions, convert blank regions to null, create `preferred_email`, remove missing IDs, fill missing regions with `Unknown`, and remove exact duplicates.

In [ ]:
# Write your solution here.

### Expected result

`customers_ready` contains four unique valid customers. C003 has region `Unknown` and a null preferred email. C001's name no longer contains surrounding spaces.

### Solution - reveal after attempting

In [ ]:
customers_ready = (
    customers
    .withColumn('customer_name', F.trim(F.col('customer_name')))
    .withColumn(
        'region',
        F.when(F.trim(F.col('region')) == '', F.lit(None))
        .otherwise(F.trim(F.col('region'))),
    )
    .withColumn('preferred_email', F.coalesce('work_email', 'personal_email'))
    .dropna(subset=['customer_id'])
    .fillna({'region': 'Unknown'})
    .dropDuplicates()
)
customers_ready.show(truncate=False)

## Key takeaway

Standardise first, apply explicit null rules, and deduplicate only when you understand which records may be discarded.

**Next:** change the grain of data with grouped calculations.